# ⚡ Marvedge Task-00046 — Standard vs Fast Pipeline Benchmark
> **Runtime → Change runtime type → GPU (T4)** before running.


In [ ]:
# Cell 1: Setup
import subprocess, os
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode==0 else '⚠️ No GPU — change runtime type!')
os.system('apt-get install -qq ffmpeg libsm6 libxext6')
os.system('pip install -q scenedetect[opencv] scipy tqdm opencv-python-headless')
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Clone repo
import os
BRANCH = 'feat/task-43-center-crop-fallback'
if not os.path.exists('/content/marvedge'):
    os.system(f'git clone -b {BRANCH} --depth 1 https://github.com/Marvedge/marvedge.git /content/marvedge')
os.chdir('/content/marvedge')
print('✅ Repo ready')
os.system('ls scripts/ml/')

In [ ]:
# Cell 3: Download YuNet face detector
import urllib.request, os
YUNET_DIR = '/content/marvedge/model/faceDetector/dnn'
YUNET_PATH = f'{YUNET_DIR}/face_detection_yunet_2023mar.onnx'
os.makedirs(YUNET_DIR, exist_ok=True)
if not os.path.exists(YUNET_PATH):
    URL = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
    urllib.request.urlretrieve(URL, YUNET_PATH)
print('✅ YuNet detector ready')

In [ ]:
# Cell 4: Upload your video (Kapil Sharma or any panel video)
from google.colab import files
import shutil, os, subprocess
print('Upload your video file (mp4/mkv/avi):')
uploaded = files.upload()
VIDEO_PATH = None
for fname in uploaded.keys():
    dest = f'/content/marvedge/demo/benchmark_dataset/{fname}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(fname, dest)
    VIDEO_PATH = dest
dur = float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',VIDEO_PATH]).decode().strip())
size_gb = os.path.getsize(VIDEO_PATH)/1e9
print(f'✅ {os.path.basename(VIDEO_PATH)} — {dur/60:.1f} min, {size_gb:.2f} GB')

In [ ]:
# Cell 5: STANDARD pipeline (baseline)
import time, json
STD_OUT = '/content/marvedge/demo/task46_standard'
STD_RPT = f'{STD_OUT}/benchmark_report.json'
print('=' * 60)
print('  STANDARD PIPELINE — Baseline')
print('=' * 60)
t0 = time.time()
os.system(f'python scripts/ml/benchmark_preprocessing.py --videoPath {VIDEO_PATH} --savePath {STD_OUT} --reportPath {STD_RPT} --nDataLoaderThread 4 --facedetScale 0.25 --minTrack 10 --numFailedDet 100 --cropScale 0.40')
t_std = time.time() - t0
with open(STD_RPT) as f: std = json.load(f)
print(f'\n✅ Done in {t_std:.1f}s | {std["fps_throughput"]} fps | {std["realtime_ratio"]}x realtime | {std["tracks_found"]} tracks')

In [ ]:
# Cell 6: FAST pipeline (optimized)
import multiprocessing
FAST_OUT = '/content/marvedge/demo/task46_fast'
FAST_RPT = f'{FAST_OUT}/benchmark_report.json'
N_CPU = multiprocessing.cpu_count()
print('=' * 60)
print(f'  ⚡ FAST PIPELINE — stride=3, scale=0.5, workers={N_CPU}')
print('=' * 60)
t0 = time.time()
os.system(f'python scripts/ml/benchmark_preprocessing_fast.py --videoPath {VIDEO_PATH} --savePath {FAST_OUT} --reportPath {FAST_RPT} --nDataLoaderThread 4 --facedetScale 0.25 --minTrack 10 --numFailedDet 100 --cropScale 0.40 --detectionStride 3 --detectionScale 0.5 --workers {N_CPU}')
t_fast = time.time() - t0
with open(FAST_RPT) as f: fast = json.load(f)
print(f'\n✅ Done in {t_fast:.1f}s | {fast["fps_throughput"]} fps | {fast["realtime_ratio"]}x realtime | {fast["tracks_found"]} tracks')

In [ ]:
# Cell 7: Comparison Report
speedup     = fast['realtime_ratio'] / max(std['realtime_ratio'], 0.01)
fps_speedup = fast['fps_throughput'] / max(std['fps_throughput'], 0.01)
print()
print('╔══════════════════════════════════════════════════════════════════╗')
print('║              TASK-00046 BENCHMARK COMPARISON                     ║')
print('╠═══════════════════════════╦══════════════════╦══════════════════╣')
print('║  METRIC                   ║  STANDARD        ║  ⚡ FAST          ║')
print('╠═══════════════════════════╬══════════════════╬══════════════════╣')
print(f'║  Total time               ║  {std["total_wall_time_sec"]:>10.1f}s     ║  {fast["total_wall_time_sec"]:>10.1f}s     ║')
print(f'║  Throughput (fps)         ║  {std["fps_throughput"]:>10.1f}      ║  {fast["fps_throughput"]:>10.1f}      ║')
print(f'║  Realtime ratio           ║  {std["realtime_ratio"]:>10.2f}x     ║  {fast["realtime_ratio"]:>10.2f}x     ║')
print(f'║  Tracks found             ║  {std["tracks_found"]:>10}      ║  {fast["tracks_found"]:>10}      ║')
print('╠═══════════════════════════╩══════════════════╩══════════════════╣')
print(f'║  ⚡ SPEEDUP:  {speedup:.2f}x faster  |  {fps_speedup:.1f}x more fps               ║')
print('╚══════════════════════════════════════════════════════════════════╝')
import json, datetime
summary = {'task':'Task-00046','video':os.path.basename(VIDEO_PATH),'speedup_x':round(speedup,2),'fps_speedup_x':round(fps_speedup,2),'standard':{'total_sec':std['total_wall_time_sec'],'fps':std['fps_throughput'],'realtime_ratio':std['realtime_ratio'],'tracks':std['tracks_found']},'fast':{'total_sec':fast['total_wall_time_sec'],'fps':fast['fps_throughput'],'realtime_ratio':fast['realtime_ratio'],'tracks':fast['tracks_found']}}
with open('/content/task46_comparison.json','w') as f: json.dump(summary,f,indent=2)
from google.colab import files
files.download('/content/task46_comparison.json')
print('✅ task46_comparison.json downloaded')